# Test Notebooks

In [ ]:
# Ensure that your python environment has huggingface_hub package installed.
import torch


# Check for GPU/MPS
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using device: {device}")

Using device: cuda


In [6]:
# Path for Shared Hub - change this to match your JupyterHub's shared directory
# Examples: /home/jovyan/shared, /home/jovyan/shared_readwrite, /home/jovyan/_shared/course-name
shared_model_path = "/home/jovyan/shared-readwrite"

### Download TinyLlama 1.1B (Recommended for teaching)

In [ ]:
from huggingface_hub import HfApi, list_models
# Search for GGUF models
api = HfApi()

# Find models with "gguf" in the name, sorted by downloads
models = list(api.list_models(
    search="gguf",
    sort="downloads",
    limit=20
))

print("Top 20 GGUF models by downloads:")
print("-" * 60)
for model in models:
    print(f"{model.id}")

Top 20 GGUF models by downloads:
------------------------------------------------------------
ggml-org/tinygemma3-GGUF
xtuner/llava-llama-3-8b-v1_1-gguf
unsloth/Qwen3-Coder-Next-GGUF
ggml-org/embeddinggemma-300M-GGUF
hugging-quants/Llama-3.2-1B-Instruct-Q8_0-GGUF
ggml-org/gpt-oss-120b-GGUF
mradermacher/Trinity-Nano-Base-GGUF
lmstudio-community/gemma-3-4b-it-GGUF
unsloth/GLM-4.7-Flash-GGUF
bartowski/Meta-Llama-3.1-8B-Instruct-GGUF
ggml-org/gemma-3-12b-it-GGUF
lmg-anon/vntl-llama3-8b-v2-gguf
janhq/Jan-v3-4B-base-instruct-gguf
unsloth/Qwen3-Coder-30B-A3B-Instruct-GGUF
unsloth/gpt-oss-20b-GGUF
MaziyarPanahi/Qwen3-14B-GGUF
MaziyarPanahi/Qwen3-4B-GGUF
MaziyarPanahi/Qwen3-0.6B-GGUF
unsloth/Qwen3.5-35B-A3B-GGUF
MaziyarPanahi/Qwen3-1.7B-GGUF


In [13]:
# List files in a specific repository to find available quantizations
from huggingface_hub import list_repo_files

repo_id = "TheBloke/TinyLlama-1.1B-Chat-v1.0-GGUF"
files = list_repo_files(repo_id)

print(f"Files in {repo_id}:")
print("-" * 60)
for f in files:
    if f.endswith(".gguf"):
        print(f)

Files in TheBloke/TinyLlama-1.1B-Chat-v1.0-GGUF:
------------------------------------------------------------
tinyllama-1.1b-chat-v1.0.Q2_K.gguf
tinyllama-1.1b-chat-v1.0.Q3_K_L.gguf
tinyllama-1.1b-chat-v1.0.Q3_K_M.gguf
tinyllama-1.1b-chat-v1.0.Q3_K_S.gguf
tinyllama-1.1b-chat-v1.0.Q4_0.gguf
tinyllama-1.1b-chat-v1.0.Q4_K_M.gguf
tinyllama-1.1b-chat-v1.0.Q4_K_S.gguf
tinyllama-1.1b-chat-v1.0.Q5_0.gguf
tinyllama-1.1b-chat-v1.0.Q5_K_M.gguf
tinyllama-1.1b-chat-v1.0.Q5_K_S.gguf
tinyllama-1.1b-chat-v1.0.Q6_K.gguf
tinyllama-1.1b-chat-v1.0.Q8_0.gguf


In [14]:
import time
def stress_test_gpu(duration_seconds=30, matrix_size=15000):
    """
    Stress tests the GPU by allocating large matrices and repeatedly multiplying them.
    Increase matrix_size to use more VRAM.
    """
    if not torch.cuda.is_available():
        print("CUDA is not available. PyTorch cannot see your GPU!")
        return

    device = torch.device("cuda")
    print(f"🚀 Detected GPU: {torch.cuda.get_device_name(device)}")
    
    # 1. Fill up VRAM
    print(f"Allocating {matrix_size}x{matrix_size} matrices to max out VRAM...")
    try:
        # Creating two massive random tensors directly on the GPU
        tensor_a = torch.randn(matrix_size, matrix_size, device=device)
        tensor_b = torch.randn(matrix_size, matrix_size, device=device)
    except RuntimeError as e:
        print(f"❌ Out of Memory Error: {e}")
        print("Your GPU doesn't have enough VRAM for this matrix size. Try lowering 'matrix_size'.")
        return

    vram_used = torch.cuda.memory_allocated() / (1024**3)
    print(f"🔥 VRAM Successfully Allocated: {vram_used:.2f} GB")
    
    # 2. Max out Compute
    print(f"Starting intensive computations for {duration_seconds} seconds. Check nvidia-smi now!")
    start_time = time.time()
    iterations = 0
    
    while time.time() - start_time < duration_seconds:
        # Perform heavy computation
        result = torch.matmul(tensor_a, tensor_b)
        
        # Forces PyTorch to wait for the GPU to finish the calculation before looping.
        # Without this, PyTorch queues operations asynchronously and might not stress the GPU consistently.
        torch.cuda.synchronize() 
        iterations += 1
        
        if iterations % 10 == 0:
            elapsed = time.time() - start_time
            print(f"   ... {iterations} operations completed in {elapsed:.1f}s")

    print("\n✅ Stress test complete.")
    print(f"Total operations: {iterations}")
    print(f"Average time per massive matrix multiplication: {duration_seconds / iterations:.4f} seconds")

    # Clean up VRAM
    del tensor_a
    del tensor_b
    del result
    torch.cuda.empty_cache()

# Run the test
stress_test_gpu(duration_seconds=300, matrix_size=15000)

🚀 Detected GPU: Tesla T4
Allocating 15000x15000 matrices to max out VRAM...
🔥 VRAM Successfully Allocated: 1.68 GB
Starting intensive computations for 60 seconds. Check nvidia-smi now!
   ... 10 operations completed in 17.7s
   ... 20 operations completed in 36.2s
   ... 30 operations completed in 54.6s

✅ Stress test complete.
Total operations: 33
Average time per massive matrix multiplication: 1.8182 seconds
